In [ ]:
from langgraph_sdk import get_client
import httpx

def get_langgraph_client():
    URL = "http://127.0.0.1:2024"
    client = get_client(url=URL)
    if client is None:
        raise Exception("Failed to connect to LangGraph server")
    try:
        client.http.client.timeout = httpx.Timeout(connect=5, read=30*60, write=5*60, pool=5)
    except Exception as e:
        print(f"Failed to set timeout for LangGraph client: {e}")
        return client
    return client

async def get_assistant(client):
    try:
        assistant = await client.assistants.search(limit=25)
        return assistant
    except Exception as e:
        raise Exception(f"Failed to get assistant: {e}")
    

_thread_id = None    
async def get_or_create_thread(client):
    global _thread_id
    _thread_id = None  
    if _thread_id is None:
        try:
            _thread_id = await client.threads.create()
            _thread_id = _thread_id['thread_id']
            print(f"Created thread: {_thread_id}")
        except Exception as e:
            print(f"Failed to create thread: {e}")
            return None
    return _thread_id


: 

In [25]:
from langchain_core.messages import HumanMessage

async def run_jd_generator(client, input_string, _thread_id):
    input_data = {
        "messages": [
            HumanMessage(content=input_string)
        ]
    }
    selected_agent = "33f5bbdc-e7a1-562f-b4e5-b620117e1da5"
    final_state = await client.runs.wait(
        config={"recursion_limit": 200},
        thread_id=_thread_id,
        assistant_id=selected_agent,
        input=input_data
    )
    return final_state


In [26]:
client = get_langgraph_client()

In [27]:
assistants = await get_assistant(client)
for assistant in assistants:
    print(f"Assistant name: {assistant['graph_id']}, ID: {assistant['assistant_id']}")

Assistant name: business_context_analyzer, ID: 7dec50d8-eb0a-5769-94e1-83e86449f127
Assistant name: job_description_generator, ID: 33f5bbdc-e7a1-562f-b4e5-b620117e1da5


In [32]:
_thread_id = await get_or_create_thread(client)
input_string = f"""
  {{
  "company_name": "Innovatech Solutions",
  "company_description": "Innovatech Solutions is a rapidly growing SaaS company specializing in AI-driven platforms that help businesses optimize their workflows and enhance customer engagement.",
  "job_title": "Full-Stack Software Engineer",
  "required_experience": "3+ years of professional software development experience in both frontend and backend technologies.",
  "required_technical_skills": [
    "Proficiency in JavaScript and React.js",
    "Strong backend experience with Node.js and Express",
    "Familiarity with databases such as PostgreSQL and MongoDB",
    "Experience with RESTful APIs and microservices architecture",
    "Version control with Git and CI/CD pipelines (e.g., GitHub Actions, Jenkins)",
    "Cloud deployment experience (AWS, Azure, or GCP)"
  ],
  "soft_skills": [
    "Strong problem-solving and analytical thinking",
    "Excellent communication and teamwork abilities",
    "Adaptability in a fast-paced agile environment",
    "Attention to detail and a proactive mindset"
  ],
  "working_conditions": "Hybrid work model (3 days on-site, 2 days remote); flexible working hours; modern office in the Melbourne CBD with ergonomic workstations and a collaborative environment.",
  "education_requirements": "Bachelor's degree in Computer Science, Software Engineering, or a related field. A Master's degree or relevant certifications (e.g., AWS Certified Developer) is a plus.",
  "key_responsibilities": [
    "Design, develop, and maintain scalable web applications using React and Node.js",
    "Collaborate with cross-functional teams including product managers, designers, and QA engineers",
    "Write clean, maintainable, and efficient code following best practices",
    "Participate in code reviews and mentor junior developers",
    "Identify and troubleshoot software issues and performance bottlenecks",
    "Contribute to architectural decisions and the overall technical direction of the product"
  ]
}}
"""

jd_output = await run_jd_generator(client, input_string, _thread_id)

Created thread: 72596ca8-6143-46e9-89dd-60af35e74f21


In [45]:
print(jd_output["formatted_job_description"]['content'])

**Job Title:** Full-Stack Software Engineer

---

**About the Company**  
Innovatech Solutions is a forward-thinking technology company specializing in AI-driven SaaS platforms. We are dedicated to delivering innovative, scalable software solutions that empower businesses to harness the power of artificial intelligence. Our collaborative and dynamic environment fosters growth, creativity, and continuous learning.

---

**Job Summary**  
We are seeking a talented Full-Stack Software Engineer to join our development team. In this role, you will design, develop, and maintain cutting-edge web applications, contributing to the evolution of our AI-powered SaaS platform. The ideal candidate will bring at least 3 years of full-stack experience, a passion for technology, and a collaborative spirit.

---

**Key Responsibilities**  
- Design, develop, and maintain scalable web applications using React.js and Node.js.  
- Collaborate with cross-functional teams including product managers, designer

In [1]:
from langchain_core.tools import tool
from typing import Optional


from langchain_mcp_adapters.client import MultiServerMCPClient
from langgraph.prebuilt import create_react_agent
from dotenv import load_dotenv

load_dotenv()

def get_mcp_client_tools(
    url="https://mcp.context7.com/mcp", 
    transport="streamable_http"
):
    client = MultiServerMCPClient(
        {
            "context7": {
                "url": url,
                "transport": transport,
            }
        }
    )
    tools = client.get_tools()
    return tools

@tool
async def get_databricks_documentation(
    library_id: str = "/websites/learn_microsoft-en-us-azure-databricks", 
    topic: Optional[str] = None,
    tokens: int = 10000
) -> str:
    """Get Azure Databricks documentation on a specific topic.
    
    Args:
        topic: Topic to focus on (e.g., 'machine learning', 'data engineering', 'JSON functions')
        tokens: Maximum number of tokens to retrieve (default: 10000)
        
    Returns:
        String containing the documentation content
    """

    tools = await get_mcp_client_tools()
    doc_retriever = tools[1] 
    result = await doc_retriever.coroutine(
        context7CompatibleLibraryID=library_id,
        topic=topic,
        tokens=tokens
    )
    if isinstance(result, tuple):
        return result[0]
    return str(result)

In [2]:
final_agent = create_react_agent("openai:gpt-4.1-nano", [get_databricks_documentation])

# Test the agent
final_response = await final_agent.ainvoke({
    "messages": "JSON convert function"
})

final_response

{'messages': [HumanMessage(content='JSON convert function', additional_kwargs={}, response_metadata={}, id='f1efc546-851e-4402-b062-8906a350e4a3'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_cqyngDygBGde0w5FA5G0Q1dg', 'function': {'arguments': '{"topic":"JSON functions"}', 'name': 'get_databricks_documentation'}, 'type': 'function'}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 19, 'prompt_tokens': 145, 'total_tokens': 164, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4.1-nano-2025-04-14', 'system_fingerprint': 'fp_c4c155951e', 'id': 'chatcmpl-CASQ2QeeoIW9UFxC0Hb7LecQH8PyC', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='run--e8045c0c-0daf-4056-ba80-5202b192055d-0', tool_calls=[{'name': 'get_databricks_documentation', 

In [45]:
final_response

{'messages': [HumanMessage(content='Get Azure Databricks documentation about JSON functions and show me how to use them', additional_kwargs={}, response_metadata={}, id='cd3576c2-1949-4495-a295-0afe5197939c'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_w1OeEYMVLW6mVoFEFbqc6IRp', 'function': {'arguments': '{"topic": "JSON functions in Azure Databricks"}', 'name': 'get_databricks_documentation'}, 'type': 'function'}, {'id': 'call_5PjsSQK5gXftYMbYr5gKVCtu', 'function': {'arguments': '{"topic": "Usage examples of JSON functions in Azure Databricks"}', 'name': 'get_databricks_documentation'}, 'type': 'function'}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 67, 'prompt_tokens': 134, 'total_tokens': 201, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4.1-nano-

In [ ]:
# Fix: Handle the tuple return from MCP tools
async def get_databricks_docs_for_agent(topic: Optional[str] = None, tokens: Optional[int] = 10000):
    """
    Fetches Azure Databricks documentation directly.
    Returns only the content string for compatibility with React agent.
    """
    result = await tools[1].coroutine(
        context7CompatibleLibraryID="/websites/learn_microsoft-en-us-azure-databricks",
        topic=topic,
        tokens=tokens
    )
    
    # MCP tools return (content, artifacts) - we just want the content
    if isinstance(result, tuple):
        return result[0]  # Return just the content string
    return result

# Create the agent-compatible tool
from pydantic import BaseModel, Field

class DatabricksInput(BaseModel):
    topic: Optional[str] = Field(None, description="Topic to focus on (e.g., 'machine learning', 'data engineering')")
    tokens: Optional[int] = Field(10000, description="Max tokens to retrieve")

databricks_agent_tool = StructuredTool(
    name="get_databricks_docs",
    description="Get Azure Databricks documentation on a specific topic",
    coroutine=get_databricks_docs_for_agent,
    args_schema=DatabricksInput
)

print("Created agent-compatible tool")

# Test it directly first
test_direct = await databricks_agent_tool.ainvoke({"topic": "JSON functions"})
print("Direct test result type:", type(test_direct))
print("Direct test preview:", str(test_direct)[:100] + "..." if len(str(test_direct)) > 100 else str(test_direct))

In [ ]:
# Test the fixed version
test_result_fixed = await databricks_tool_fixed.ainvoke({"topic": "JSON function"})
print("Fixed test result type:", type(test_result_fixed))
print("Fixed test result preview:", str(test_result_fixed)[:200] + "..." if len(str(test_result_fixed)) > 200 else str(test_result_fixed))

In [49]:
# Complete Setup Function - Run this cell after notebook restart
from langchain_mcp_adapters.client import MultiServerMCPClient
from langgraph.prebuilt import create_react_agent
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from dotenv import load_dotenv
from typing import Optional
import asyncio

async def setup_databricks_agent():
    """Complete setup for Databricks documentation agent - run after notebook restart"""
    
    print("🔧 Setting up Databricks Documentation Agent...")
    
    # Load environment
    load_dotenv()
    
    # Step 1: Create MCP client and get tools
    print("📡 Connecting to Context7 MCP server...")
    client = MultiServerMCPClient({
        "context7": {
            "url": "https://mcp.context7.com/mcp",
            "transport": "streamable_http",
        }
    })
    
    tools = await client.get_tools()
    print(f"✅ Retrieved {len(tools)} tools from MCP server")
    
    # Step 2: Create custom Databricks tool
    print("🛠️ Creating custom Databricks documentation tool...")
    
    @tool
    async def get_databricks_documentation(topic: Optional[str] = None, tokens: int = 10000) -> str:
        """Get Azure Databricks documentation on a specific topic.
        
        Args:
            topic: Topic to focus on (e.g., 'machine learning', 'data engineering', 'JSON functions')
            tokens: Maximum number of tokens to retrieve (default: 10000)
            
        Returns:
            String containing the documentation content
        """
        result = await tools[1].coroutine(
            context7CompatibleLibraryID="/websites/learn_microsoft-en-us-azure-databricks",
            topic=topic,
            tokens=tokens
        )
        
        # Handle tuple return from MCP tools
        if isinstance(result, tuple):
            return result[0]  # Return just the content string
        return str(result)
    
    # Step 3: Create React agent
    print("🤖 Creating React agent...")
    databricks_agent = create_react_agent("openai:gpt-4.1-nano", [get_databricks_documentation])
    
    print("✅ Setup complete! Ready to use.")
    
    return databricks_agent, get_databricks_documentation

# Run setup and store results
databricks_agent, databricks_tool = await setup_databricks_agent()

🔧 Setting up Databricks Documentation Agent...
📡 Connecting to Context7 MCP server...
✅ Retrieved 2 tools from MCP server
🛠️ Creating custom Databricks documentation tool...
🤖 Creating React agent...
✅ Setup complete! Ready to use.


In [50]:
databricks_response = await databricks_agent.ainvoke({
    "messages": "JSON convert function"
})

databricks_response

{'messages': [HumanMessage(content='JSON convert function', additional_kwargs={}, response_metadata={}, id='9637c325-085c-493c-a95e-06390a71392a'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_Y6kA6Xcii6HABMjZmzixtdjG', 'function': {'arguments': '{"topic":"JSON functions"}', 'name': 'get_databricks_documentation'}, 'type': 'function'}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 19, 'prompt_tokens': 121, 'total_tokens': 140, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4.1-nano-2025-04-14', 'system_fingerprint': 'fp_c4c155951e', 'id': 'chatcmpl-CASJ0VJk3h0BtjJBorbTVRnM207cj', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='run--19720211-e96e-44fc-aca0-40868f201175-0', tool_calls=[{'name': 'get_databricks_documentation', 

In [ ]:
# Complete End-to-End Workflow Test for Custom Databricks Tool
import asyncio
from langchain_core.messages import HumanMessage

async def test_databricks_workflow():
    """Complete workflow test for the custom Databricks documentation tool"""
    
    print("🚀 Starting End-to-End Databricks Documentation Workflow Test")
    print("=" * 60)
    
    # Step 1: Test the tool directly
    print("\n📋 Step 1: Testing tool directly")
    try:
        direct_result = await get_databricks_documentation.ainvoke({
            "topic": "machine learning pipelines", 
            "tokens": 5000
        })
        print("✅ Direct tool test successful")
        print(f"📄 Content length: {len(direct_result)} characters")
        print(f"🔍 Content preview: {direct_result[:150]}...")
    except Exception as e:
        print(f"❌ Direct tool test failed: {e}")
        return False
    
    # Step 2: Create React agent with the tool
    print("\n🤖 Step 2: Creating React agent with custom tool")
    try:
        databricks_agent = create_react_agent("openai:gpt-4.1-nano", [get_databricks_documentation])
        print("✅ React agent created successfully")
    except Exception as e:
        print(f"❌ Agent creation failed: {e}")
        return False
    
    # Step 3: Test various queries with the agent
    test_queries = [
        "What are JSON functions in Databricks?",
        "How do I use machine learning in Azure Databricks?",
        "Show me data engineering best practices in Databricks",
        "What are the key features of Databricks SQL?"
    ]
    
    print(f"\n🔍 Step 3: Testing {len(test_queries)} different queries")
    
    for i, query in enumerate(test_queries, 1):
        print(f"\n📝 Query {i}: {query}")
        try:
            response = await databricks_agent.ainvoke({
                "messages": [HumanMessage(content=query)]
            })
            
            # Extract the final message
            if isinstance(response, dict) and 'messages' in response:
                final_message = response['messages'][-1]
                content = final_message.content if hasattr(final_message, 'content') else str(final_message)
                print(f"✅ Query {i} successful")
                print(f"📊 Response length: {len(content)} characters")
                print(f"🎯 Response preview: {content[:200]}...")
            else:
                print(f"⚠️ Query {i} returned unexpected format: {type(response)}")
                
        except Exception as e:
            print(f"❌ Query {i} failed: {e}")
    
    # Step 4: Performance and capability summary
    print("\n📈 Step 4: Workflow Summary")
    print("=" * 40)
    print("✅ Custom tool bypasses resolve-library-id call")
    print("✅ Direct tool invocation works")
    print("✅ React agent integration successful")
    print("✅ Multiple query types handled")
    print("✅ Proper async handling implemented")
    print("✅ MCP tuple return properly handled")
    
    print(f"\n🎉 End-to-End Workflow Test COMPLETED SUCCESSFULLY!")
    return True

# Run the complete workflow test
await test_databricks_workflow()

In [31]:
# Fix: Create the structured tool properly for async functions
from langchain_core.tools import StructuredTool
from typing import Optional
import inspect

# The async function is correct, but we need to create the tool differently
async def get_databricks_docs_fixed(topic: Optional[str] = None, tokens: Optional[int] = 10000):
    """
    Fetches Azure Databricks documentation directly.
    
    Args:
        topic: Topic to focus documentation on (e.g., 'machine learning', 'data engineering')  
        tokens: Maximum number of tokens of documentation to retrieve (default: 10000)
    """
    result = await tools[1].coroutine(
        context7CompatibleLibraryID="/websites/learn_microsoft-en-us-azure-databricks",
        topic=topic,
        tokens=tokens
    )
    return result

# Create the tool using coroutine parameter instead of func
databricks_tool_fixed = StructuredTool(
    name="get_databricks_docs", 
    description="Fetches Azure Databricks documentation. Specify a topic to focus on specific areas like 'machine learning', 'data engineering', 'spark', etc.",
    coroutine=get_databricks_docs_fixed,
    args_schema=databricks_tool.args_schema  # Reuse the same schema
)

print("Created fixed Databricks tool")
print(f"Tool name: {databricks_tool_fixed.name}")
print(f"Has coroutine: {databricks_tool_fixed.coroutine is not None}")
print(f"Has func: {databricks_tool_fixed.func is not None}")

Created fixed Databricks tool
Tool name: get_databricks_docs
Has coroutine: True
Has func: False


In [32]:
# Test the current issue - this will show the problem
test_result = await databricks_tool_fixed.ainvoke({"topic": "JSON function"})
print("Test result type:", type(test_result))
print("Test result:", test_result)

Test result type: <class 'tuple'>
Test result: ('========================\nCODE SNIPPETS\n========================\nTITLE: JSON Functions\nDESCRIPTION: Functions for parsing and querying JSON data.\n\nSOURCE: https://learn.microsoft.com/en-us/azure/databricks/sql/language-manual/sql-ref-functions-builtin-alpha\n\nLANGUAGE: APIDOC\nCODE:\n```\nJSON Functions:\n\n- json_array_length(json_array)\n  - Returns the number of elements in a JSON array.\n  - Parameters:\n    - json_array: The JSON array string.\n  - Returns: The number of elements (INTEGER).\n\n- json_object_keys(json_object)\n  - Returns the keys of a JSON object as an array.\n  - Parameters:\n    - json_object: The JSON object string.\n  - Returns: An array of strings representing the keys.\n\n- json_tuple(json_string, path1, path2, ...)\n  - Extracts multiple fields from a JSON string into separate columns.\n  - Parameters:\n    - json_string: The JSON string.\n    - path1, path2, ...: Paths to the fields to extract (e.g., \

In [ ]:
from langchain_core.tools import StructuredTool
from typing import Optional

# Create a custom wrapper for the Databricks documentation tool
async def get_databricks_docs(topic: Optional[str] = None, tokens: Optional[int] = 10000):
    """
    Fetches Azure Databricks documentation directly.
    
    Args:
        topic: Topic to focus documentation on (e.g., 'machine learning', 'data engineering')
        tokens: Maximum number of tokens of documentation to retrieve (default: 10000)
    """
    # Use the original tool but with pre-configured library ID
    return await tools[1].coroutine(
        context7CompatibleLibraryID="/websites/learn_microsoft-en-us-azure-databricks",
        topic=topic,
        tokens=tokens
    )

# Create a new structured tool
databricks_tool = StructuredTool.from_function(
    func=get_databricks_docs,
    name="get_databricks_docs",
    description="Azure Databricks is a unified analytics platform designed for data analysts, data engineers, data scientists, and machine learning engineers. It provides tools for data warehousing, data engineering, and machine learning on a scalable platform."
)

# Replace the original tool in the tools list
tools_with_databricks = [databricks_tool]  # Keep resolve-library-id as tools[0]

print("Created custom Databricks documentation tool")
print(f"Tool name: {databricks_tool.name}")
print(f"Tool description: {databricks_tool.description}")

Created custom Databricks documentation tool
Tool name: get_databricks_docs
Tool description: Azure Databricks is a unified analytics platform designed for data analysts, data engineers, data scientists, and machine learning engineers. It provides tools for data warehousing, data engineering, and machine learning on a scalable platform.


In [29]:
await databricks_tool.ainvoke("JSON function")

<coroutine object get_databricks_docs at 0x119cdfbc0>

In [33]:
# Example usage: Now you can create an agent that directly uses the Databricks documentation
# without needing to call resolve-library-id first

agent_with_databricks = create_react_agent("openai:gpt-4.1-nano", databricks_tool_fixed)

# Example query that will now use the pre-configured library ID
databricks_response = await agent_with_databricks.ainvoke({
    "messages": "JSON convert function"
})

ValueError: The first argument must be a string or a callable with a __name__ for tool decorator. Got <class 'tuple'>

In [25]:
databricks_response

{'messages': [HumanMessage(content='JSON convert function in databricks', additional_kwargs={}, response_metadata={}, id='2638d98a-4ffa-4b73-b27a-a7f0090832b5'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_4alO8pyBAWmdY4iSHY2vcho2', 'function': {'arguments': '{"topic":"JSON convert function"}', 'name': 'get_databricks_docs'}, 'type': 'function'}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 19, 'prompt_tokens': 112, 'total_tokens': 131, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4.1-nano-2025-04-14', 'system_fingerprint': 'fp_c4c155951e', 'id': 'chatcmpl-CARgQlQEaNCYaVUKgP9fyaZlT6bgn', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='run--4f3f530f-34d3-4d00-a166-3aef739b5fd5-0', tool_calls=[{'name': 'get_databricks_docs